In [3]:
import csv
import pandas as pd
import re

# Define path
input_path = "funding.csv"
output_path = "biodiversity_funding_supercleaned.csv"

# Function to clean each cell
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'\s+', ' ', text)  # Remove newline, tabs, multiple spaces
    text = re.sub(r'&#\d+;', '', text)  # Remove HTML unicode references like &#8203;
    text = re.sub(r'&nbsp;|&amp;|&lt;|&gt;', ' ', text)  # Replace common HTML entities
    text = re.sub(r'[\u200b\u200c\u200d\u200e\u200f]', '', text)  # Remove zero-width chars
    text = re.sub(r':?contentReference\[.*?\]', '', text)  # Remove contentReference[...] with optional colon
    text = re.sub(r'\{index=\d+\}', '', text)  # Remove {index=#}
    return text.strip()

# Step 1: Read file manually to fix bad rows
rows = []
with open(input_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    for i, row in enumerate(reader):
        rows.append((i, row, len(row)))

# Get expected column count from header
header = rows[0][1]
expected_columns = len(header)

# Step 2: Clean and fix rows
cleaned_rows = []
for i, row, length in rows:
    if i == 0:
        cleaned_rows.append(row)
    elif length == expected_columns:
        cleaned_rows.append(row)
    elif i == 44 and length == 14:
        # Fix Row 45: merge columns 0 and 1
        fixed_row = [row[0] + row[1]] + row[2:]
        if len(fixed_row) == expected_columns:
            cleaned_rows.append(fixed_row)
    else:
        print(f"⚠️ Skipped row {i} with {length} columns: {row}")

# Step 3: Convert to DataFrame
df = pd.DataFrame(cleaned_rows[1:], columns=cleaned_rows[0])

# ✅ Step 3.5: Rename columns
df = df.rename(columns={
    "Type": "grant_type",
    "Eligibility Criteria": "org_type",
    "Geographic Focus":"geographic_focus",
    "Funding Focus":"funding_focus",
    "Funding Name":"funding_name",
    "Funder":"grant_funder",
    "Funding Amount":"funding_amount",
    "Application Deadline":"application_deadline",
    "Past Recipients (if available)":"past_recipients",
    "Gotchas":"gotchas",
    "Additional Notes":"additional_notes"

})

# Step 4: Clean each cell
df_cleaned = df.applymap(clean_text)

# Step 5: Save cleaned CSV
df_cleaned.to_csv(output_path, index=False)
print(f"✅ Super-cleaned CSV saved to: {output_path}")


✅ Super-cleaned CSV saved to: biodiversity_funding_supercleaned.csv


C:\Users\Mahek Pardeshi\AppData\Local\Temp\ipykernel_17036\2473047738.py:67: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_cleaned = df.applymap(clean_text)


In [4]:
import pandas as pd
import json

# Load the cleaned CSV
csv_path = "biodiversity_funding_supercleaned.csv"
jsonl_path = "supercleaned_for_embedding.jsonl"

# Read the CSV into a DataFrame
df = pd.read_csv(csv_path)

# Write to JSONL (one record per line)
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for record in df.to_dict(orient='records'):
        json_line = json.dumps(record, ensure_ascii=False)
        f.write(json_line + '\n')

print(f"✅ JSONL file saved as: {jsonl_path}")


✅ JSONL file saved as: supercleaned_for_embedding.jsonl
